# 11 — Public Telecom dataset validation

Public sources qualify different claims and are never pooled into one fitted
model:

- commercial RAN PM counters: real operator telemetry portability and workload;
- Microsoft optical: real optical drift and alert-rate stability;
- optical failure testbed: response to controlled physical failures.

RAN and Microsoft mappings require explicit human approval because native
column names, units and aggregation semantics must not be guessed. Microsoft
outage days were removed, so fault recall is unsupported. The testbed does not
provide component-localisation truth, so localisation accuracy is unsupported.


## 1. Setup and evidence policy


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import copy

import pandas as pd
import yaml
from IPython.display import display

from telco_anomaly.adapters import (
    build_microsoft_optical_pack,
    build_optical_failure_pack,
    build_ran_pm_pack,
    inspect_microsoft_optical,
    inspect_optical_failure,
    inspect_ran_pm,
    microsoft_mapping_template,
    ran_mapping_template,
)
from telco_anomaly.contract import build_canonical
from telco_anomaly.io import (
    load_config,
    read_json,
    resolve_data_root,
    resolve_dataset_source,
)

DATA_ROOT = resolve_data_root()
DATASETS = load_config("dataset", project_root=PROJECT_ROOT)
SELECTED = os.getenv("PUBLIC_DATASET", "all")
allowed = {"all", "ran_pm", "microsoft_optical", "optical_failure"}
if SELECTED not in allowed:
    raise ValueError(f"PUBLIC_DATASET must be one of {sorted(allowed)}")

claims = pd.DataFrame([
    {
        "dataset": "ran_pm",
        "supported": "portability, cadence/quality handling, alert workload",
        "unsupported": "PON localisation and PON fault recall",
    },
    {
        "dataset": "microsoft_optical",
        "supported": "real optical drift and alert-rate stability",
        "unsupported": "fault recall (outage days removed)",
    },
    {
        "dataset": "optical_failure",
        "supported": "score response and event detection on controlled failures",
        "unsupported": "production prevalence and component localisation accuracy",
    },
])
display(claims)
assert DATASETS["rules"]["pool_raw_datasets"] is False
assert DATASETS["rules"]["fit_one_model_across_metric_packs"] is False


## 2. Inventory available public sources


In [ ]:
def optional_source(name):
    try:
        return resolve_dataset_source(name, data_root=DATA_ROOT, project_root=PROJECT_ROOT)
    except FileNotFoundError:
        return None


sources = {
    name: optional_source(name)
    for name in ("ran_pm", "microsoft_optical", "optical_failure")
    if SELECTED in {"all", name}
}
display(pd.DataFrame([
    {"dataset": name, "source": str(path) if path else "not downloaded", "available": path is not None}
    for name, path in sources.items()
]))


## 3. Create mapping drafts—never inferred mappings

For RAN and Microsoft, this cell writes a draft YAML beside the external data
root. Inspect source documentation, fill native fields and units, then set
`mapping_review_status: approved`. Until that line is present, no pack is
built.


In [ ]:
MAPPING_ROOT = DATA_ROOT / "mappings"
MAPPING_ROOT.mkdir(parents=True, exist_ok=True)

def review_mapping(name, source, inspect_function, template_function):
    inventory = inspect_function(source)
    display(pd.Series({
        "files": inventory.get("file_count", len(inventory.get("files", []))),
        "columns": inventory.get("columns", []),
    }, name=name).to_frame())
    path = MAPPING_ROOT / f"{name}.yml"
    if not path.exists():
        draft = template_function()
        draft["mapping_review_status"] = "draft"
        path.write_text(yaml.safe_dump(draft, sort_keys=False), encoding="utf-8")
        print("Created mapping draft:", path)
    mapping = yaml.safe_load(path.read_text(encoding="utf-8"))
    return path, mapping


mappings = {}
if sources.get("ran_pm"):
    mappings["ran_pm"] = review_mapping(
        "ran_pm", sources["ran_pm"], inspect_ran_pm, ran_mapping_template
    )
if sources.get("microsoft_optical"):
    mappings["microsoft_optical"] = review_mapping(
        "microsoft_optical", sources["microsoft_optical"],
        inspect_microsoft_optical, microsoft_mapping_template,
    )
if sources.get("optical_failure"):
    display(pd.Series(inspect_optical_failure(sources["optical_failure"]), name="optical_failure"))


## 4. Build only reviewed, separate packs


In [ ]:
build_status = []
licence_acknowledged = {
    "microsoft_optical": os.getenv("ACKNOWLEDGE_MICROSOFT_DATA_TERMS", "0") == "1",
    "optical_failure": os.getenv("ACKNOWLEDGE_OPTICAL_FAILURE_TERMS", "0") == "1",
}

def build_reviewed(name, builder):
    mapping_path, mapping = mappings[name]
    mapping = dict(mapping)
    if mapping.pop("mapping_review_status", "draft") != "approved":
        build_status.append({"dataset": name, "status": "STOP — mapping review required"})
        return
    if name in licence_acknowledged and not licence_acknowledged[name]:
        build_status.append({"dataset": name, "status": "STOP — source terms not acknowledged"})
        return
    pack_root = DATA_ROOT / "prepared" / name / DATASETS["datasets"][name]["output_run_id"]
    core_root = DATA_ROOT / "core" / name / DATASETS["datasets"][name]["output_run_id"]
    if not pack_root.exists():
        builder(sources[name], pack_root, mapping)
    if not core_root.exists():
        build_canonical(pack_root, core_root, include_evaluation=False)
    build_status.append({"dataset": name, "status": "pack and truth-unmounted core ready"})


if "ran_pm" in mappings:
    build_reviewed("ran_pm", build_ran_pm_pack)
if "microsoft_optical" in mappings:
    build_reviewed("microsoft_optical", build_microsoft_optical_pack)
if sources.get("optical_failure"):
    name = "optical_failure"
    if not licence_acknowledged[name]:
        build_status.append({
            "dataset": name, "status": "STOP — source terms not acknowledged",
        })
    else:
        pack_root = DATA_ROOT / "prepared" / name / DATASETS["datasets"][name]["output_run_id"]
        core_root = DATA_ROOT / "core" / name / DATASETS["datasets"][name]["output_run_id"]
        if not pack_root.exists():
            build_optical_failure_pack(sources[name], pack_root, include_evaluation=True)
        if not core_root.exists():
            build_canonical(pack_root, core_root, include_evaluation=False)
        build_status.append({
            "dataset": name,
            "status": "core ready; PACK-EVAL reserved for event detection only",
        })

status = pd.DataFrame(build_status)
display(status if len(status) else pd.DataFrame({"status": ["No public source downloaded"]}))


## 5. Qualification protocol


In [ ]:
protocol = pd.DataFrame([
    {
        "step": 1,
        "rule": "For approved RAN or Microsoft packs, set TELCO_DATASET to the dataset name and run Notebooks 04–06; each dataset fits its own calibration reference.",
    },
    {
        "step": 2,
        "rule": "For RAN and Microsoft, report coverage, score stability and incident workload—not false-positive rate without labels.",
    },
    {
        "step": 3,
        "rule": "Keep the optical testbed in this benchmark notebook; its episode-specific failure protocol is not interchangeable with the longitudinal 04–06 workflow.",
    },
    {
        "step": 4,
        "rule": "Never pool raw metrics, thresholds or fitted models across PON, RAN and backbone optical packs.",
    },
])
display(protocol)

print("Example: TELCO_DATASET=ran_pm, then run 04, 05 and 06 after the RAN pack is ready.")
print("Do not run label-based Notebook 07 for RAN or Microsoft data.")

print("PASS — public evidence is separated by claim and mapping review")
print("Next: 12_INFERENCE_DEMO_AND_MODEL_CARD.ipynb")
